In [40]:
def gradient_log(traj, idx=0):
    """
    For step `idx`, show every gradient it received (backprop2 schema).
    grad entries: {'from': ..., 'gradient': ..., 'jacobian': ...}
    No separate attribution list — jacobian plays that role.
    """
    steps    = traj['steps']
    logs     = traj['logs']
    metadata = traj['metadata']
    step_map = {s['step_idx']: s for s in steps}

    # index logs by output_step_idx for fast lookup
    log_by_output = {log['output_step_idx']: log for log in logs}

    target = step_map[idx]
    grads  = target.get('grad', [])

    W = 70
    lines = []
    lines.append('=' * W)
    lines.append(f"STEP {idx}  [{target['role']}]  requires_grad={target.get('requires_grad', True)}")
    lines.append(f"inputs={target['input_steps']}  outputs={target['output_steps']}")
    lines.append('=' * W)
    lines.append('CONTENT:')
    lines.append(target['content'].strip())
    lines.append('')
    lines.append(f'GRADIENTS RECEIVED: {len(grads)}')
    lines.append('')

    for i, g in enumerate(grads):
        src      = g['from']
        gradient = g.get('gradient', '').strip()
        jacobian = g.get('jacobian', 'N/A')

        lines.append('┌' + '─' * (W - 1))
        if src == 'output_loss':
            lines.append(f'│ PATH {i+1}: LOSS SEED')
            lines.append('│')
            lines.append('│ [downstream grad of loss node]')
            lines.append(f'│   {gradient}')
        else:
            src_step = step_map.get(src, {})
            src_role = src_step.get('role', '?')

            log           = log_by_output.get(src, {})
            co_input_idxs = [j for j in log.get('input_step_idxs', []) if j != idx]

            y_downstream = _extract_downstream_grad_from_prompt(log)

            lines.append(f'│ PATH {i+1}: step {src} [{src_role}] → step {idx}')
            lines.append('│')

            # y's content
            lines.append(f'│ ── y (step {src}) content:')
            for chunk in src_step.get('content', '').strip().split('\n'):
                lines.append(f'│   {chunk}')
            lines.append('│')

            # y's downstream grad
            lines.append(f'│ ── y\'s downstream grad (dL/dy):')
            if y_downstream:
                for chunk in y_downstream.strip().split('\n'):
                    lines.append(f'│   {chunk}')
            else:
                lines.append(f'│   (not available)')
            lines.append('│')

            # co-inputs
            lines.append(f'│ ── co-inputs to y (other inputs seen by backward LLM):')
            if co_input_idxs:
                for ci in co_input_idxs:
                    co = step_map.get(ci, {})
                    preview = co.get('content', '')[:120].replace('\n', ' ')
                    lines.append(f'│   step {ci} [{co.get("role", "?")}]: {preview}...')
            else:
                lines.append(f'│   (none — x was the only input to y)')
            lines.append('│')

            # jacobian (dy/dx) replaces attribution
            lines.append(f'│ ── jacobian (dy/dx): {jacobian}')
            lines.append(f'│ ── gradient (dL/dx):')
            for chunk in gradient.split('\n'):
                lines.append(f'│   {chunk}')

        lines.append('└' + '─' * (W - 1))
        lines.append('')

    lines.append(f"GT : {metadata.get('ground_truth', '?')}")
    lines.append(f"Q  : {metadata.get('question', '?')}")
    lines.append('=' * W)
    return '\n'.join(lines)


def log_all_gradients(traj):
    """
    Print forward + backward views of all gradients (backprop2 schema).
    grad entries: {'from': ..., 'gradient': ..., 'jacobian': ...}
    """
    steps    = traj['steps']
    metadata = traj['metadata']

    SEP = "\n\n---\n\n"
    chat_content = SEP.join([
        f"STEP {i} [{entry.get('role', 'Unknown Agent')}]: {entry.get('content', '')}"
        for i, entry in enumerate(steps)
    ])

    W = 70
    lines = []
    lines.append('=' * W)
    lines.append(f"Q  : {metadata.get('question', '?')}")
    lines.append(f"GT : {metadata.get('ground_truth', '?')}")
    lines.append('=' * W)
    lines.append("FORWARD VIEW: The sequence of steps in the trajectory")
    lines.append('=' * W)
    lines.append(chat_content)
    lines.append('=' * W)
    lines.append('BACKWARD VIEW: How textual gradients flow.')
    lines.append('=' * W)

    for step in reversed(steps):
        idx   = step['step_idx']
        grads = step.get('grad', [])

        lines.append(f"\nSTEP {idx} [{step['role']}]  {step['input_steps']} → {step['output_steps']}")
        lines.append('-' * W)

        if not grads:
            lines.append('  (no gradient)')
            continue

        for g in grads:
            src      = g.get('from', 'loss')
            gradient = g.get('gradient', '').replace('\n', ' ')
            jacobian = g.get('jacobian', 'N/A')
            src_label = 'LOSS' if src == 'output_loss' else f'step {src}'

            lines.append(f"  dL/dy (downstream gradient) from {src_label}")
            lines.append(f"  dy/dx (jacobian): {jacobian}")
            lines.append(f"  dL/dx (total gradient): {gradient}")
            lines.append('')

    lines.append('=' * W)
    lines.append(f"Human annotation: Step {metadata.get('mistake_step')} - generated by {metadata.get('mistake_agent')}. - is the decisive error step.")
    lines.append(f"The reason is: {metadata.get('mistake_reason')}")
    lines.append('=' * W)
    return '\n'.join(lines)

In [41]:
command = """You are given a failed multi-agent trajectory with:
- A human-annotated decisive error step and the reason for annotation.
- Textual gradients computed via backward pass through the trajectory graph.
Perform the following analysis:
1. **Ideal vs. Actual Gradient at the Labeled Step**
   Compare the human's annotation (which step failed and why) against the textual gradient that step actually received. What information is present in the human's reasoning but absent from the gradient, and vice versa?
   Trace along the chain of gradients and act like an debugger, explain why the textual gradient is different from human's annotation at the target step.
2. **Gradient Quality Audit**
   For each step that received a gradient, assess the quality of the criticism of that step. 
   | Step | Detailed Assessment | Overall assessment (Score: ?/10) |
3. **Attribution Application**
   Explain whether the textual gradients and jacobians can be helpful in identifying the decisive error steps by further post-hoc processing.
4. **Pathology Patterns**
   List any systematic issues observed in this trajectory's gradients that may generalize to other trajectories (e.g., recurring biases, information loss across hops).

Only search following files to support your analysis:
- TextGrad
- agent_grad.core, agent_grad.backward, cli.backprop
"""
print(command)

You are given a failed multi-agent trajectory with:
- A human-annotated decisive error step and the reason for annotation.
- Textual gradients computed via backward pass through the trajectory graph.
Perform the following analysis:
1. **Ideal vs. Actual Gradient at the Labeled Step**
   Compare the human's annotation (which step failed and why) against the textual gradient that step actually received. What information is present in the human's reasoning but absent from the gradient, and vice versa?
   Trace along the chain of gradients and act like an debugger, explain why the textual gradient is different from human's annotation at the target step.
2. **Gradient Quality Audit**
   For each step that received a gradient, assess the quality of the criticism of that step. 
   | Step | Detailed Assessment | Overall assessment (Score: ?/10) |
3. **Attribution Application**
   Explain whether the textual gradients and jacobians can be helpful in identifying the decisive error steps by furth

In [42]:
import json
from pathlib import Path
from utils.common import _load_json_data, _get_sorted_json_files

def satisfy_length(dir, filename, lo=19, hi=40):
    filepath = dir / filename
    data = _load_json_data(filepath)
    num_steps = len(data['steps'])
    return lo <= num_steps <= hi

lo, hi = 19, 40
debug_dir = Path('outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted')
result_files = _get_sorted_json_files(debug_dir)
result_files = [x for x in result_files if satisfy_length(debug_dir, x, lo, hi)]
print(f"files with number of steps in {lo}-{hi}: {result_files}")

debug_file = debug_dir / result_files[0]
traj = _load_json_data(debug_file)
print(f'debugging: {debug_file}: {len(traj['steps'])} steps')

# gradient_ingredients = gradient_log(traj, idx=88)
gradient_ingredients = command
gradient_ingredients += log_all_gradients(traj)
with open('outputs/debug.txt', 'w') as f:
    f.write(gradient_ingredients)

files with number of steps in 19-40: ['1.json', '5.json', '7.json', '12.json', '14.json', '16.json', '17.json', '18.json', '21.json', '22.json', '25.json', '26.json', '28.json', '31.json', '42.json', '45.json', '52.json', '53.json', '54.json']
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/1.json: 30 steps


In [44]:
type(Path("01.json").stem)

str

In [47]:
import json
from pathlib import Path
from utils.common import _load_json_data, _get_sorted_json_files

def satisfy_length(dir, filename, lo=19, hi=40):
    filepath = dir / filename
    data = _load_json_data(filepath)
    num_steps = len(data['steps'])
    return lo <= num_steps <= hi

lo, hi = 19, 40
debug_dir = Path('outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted')
result_files = _get_sorted_json_files(debug_dir)
result_files = [x for x in result_files if satisfy_length(debug_dir, x, lo, hi)]

for filepath in result_files: 
    debug_file = debug_dir / filepath
    traj = _load_json_data(debug_file)
    print(f'debugging: {debug_file}: {len(traj['steps'])} steps')

    # gradient_ingredients = gradient_log(traj, idx=88)
    gradient_ingredients = command
    gradient_ingredients += log_all_gradients(traj)
    output_file = Path(filepath).stem + ".txt"
    output_path = Path("outputs/diagnosis") / output_file
    with open(output_path, 'w') as f:
        f.write(gradient_ingredients)

debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/1.json: 30 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/5.json: 21 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/7.json: 26 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/12.json: 21 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/14.json: 33 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/16.json: 22 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/17.json: 38 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/18.json: 32 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/21.json: 26 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/22.json: 25 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/25.json: 21 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/26.json: 34 steps
debugging: outputs/gpt-oss-20b-wgt/agent-grad/hand-crafted/28.json: 33 steps
de

In [50]:
import pandas as pd
df = pd.read_csv("outputs/gpt-oss-20b_sweep-by-length.tsv", sep='\t')

In [65]:
length_levels = {k: v for k, v in zip(
    ["short", "medium", "long", "all"],
    df["trajectory_length"].to_list()[:4]
)}
length_levels

{'short': '5-23', 'medium': '24-58', 'long': '59+', 'all': 'all'}

In [76]:
sub_report = df[
    (df["k"] == 1) 
    & (df["trajectory_length"] == length_levels["short"])
].to_csv(sep='\t', index=False)
print(sub_report)

model	strategy	subset	trajectory_length	k	total	agent_acc	step_acc
gpt-oss-20b	all-at-once	hand-crafted	5-23	1	20	50.0	55.0
gpt-oss-20b	step-by-step	hand-crafted	5-23	1	20	40.0	40.0
gpt-oss-20b-v2	agent-grad	hand-crafted	5-23	1	19	0.0	5.26



In [24]:
# import json
# from pathlib import Path
# from utils.common import _load_json_data, _get_sorted_json_files
# debug_dir = Path('outputs/gpt-oss-20b/agent-grad/short-context')
# result_files = _get_sorted_json_files(debug_dir)
# debug_file = debug_dir / result_files[7]
# traj = _load_json_data(debug_file)
# print(f'debugging: {debug_file}: {len(traj['steps'])} steps')

# # gradient_ingredients = gradient_log(traj, idx=88)
# gradient_ingredients = command
# gradient_ingredients += log_all_gradients(traj)
# with open('outputs/debug.txt', 'w') as f:
#     f.write(gradient_ingredients)

debugging: outputs/gpt-oss-20b/agent-grad/short-context/32.json: 13 steps
